"""05_feature_engineering.ipynb
Tạo các đặc trưng (features) cho bài toán dự đoán churn.
- Đầu vào: Dữ liệu đã được làm sạch và gộp (merged_orders.parquet)
- Đầu ra: Bảng features ở cấp độ khách hàng (customer_unique_id)
"""

In [8]:
import pandas as pd
import numpy as np
from churn_prediction.paths import PROCESSED_DIR, FEATURES_DIR, INTERIM_CLI_DIR
import warnings
warnings.filterwarnings('ignore')

In [7]:
# Dữ liệu đã được gom nhóm theo customer
customer_df = pd.read_parquet(PROCESSED_DIR / 'customer_level.parquet')

# Dữ liệu gốc để tính các features
df_merged = pd.read_parquet(INTERIM_CLI_DIR / 'merged_orders.parquet')

print(f"   Customer level: {customer_df.shape}")
print(f"   Merged orders: {df_merged.shape}")

   Customer level: (96096, 32)
   Merged orders: (99441, 26)


In [14]:
print(df_merged.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'total_price', 'avg_price', 'num_products', 'total_freight', 'avg_freight', 'unique_products', 'unique_sellers', 'total_payment', 'max_installments', 'main_payment_type', 'review_score', 'num_comment_messages', 'num_comment_titles', 'days_to_answer', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'delivery_time_days', 'estimated_delivery_days', 'delivery_delay']


### 2. NHÓM 1: TRẢI NGHIỆM GIAO HÀNG (LOGISTICS)

In [16]:
# Tính thời gian giao hàng (đã có trong merged orders)
# delivery_days = ngày giao hàng thực tế - ngày mua
df_merged['delivery_time_days'] = (
    df_merged['order_delivered_customer_date'] - df_merged['order_purchase_timestamp']
).dt.days

# Thời gian giao hàng dự kiến
df_merged['estimated_delivery_days'] = (
    df_merged['order_estimated_delivery_date'] - df_merged['order_purchase_timestamp']
).dt.days

# Độ trễ giao hàng (ngày giao trễ so với dự kiến)
df_merged['delivery_delay'] = df_merged['delivery_time_days'] - df_merged['estimated_delivery_days']

# Aggregation theo customer
delivery_features = df_merged.groupby('customer_unique_id').agg({
    'delivery_time_days': ['mean', 'std', 'max'],
    'estimated_delivery_days': 'mean',
    'delivery_delay': ['mean', 'std', 'max'],
    'avg_freight': ['mean', 'sum'], 
    'total_freight': 'mean',
    
}).reset_index()

delivery_features.columns = [
    'customer_unique_id',
    'avg_delivery_time', 'std_delivery_time', 'max_delivery_time',
    'avg_estimated_delivery',
    'avg_delivery_delay', 'std_delivery_delay', 'max_delivery_delay',
    'avg_freight_per_order', 'total_freight_from_avg',
    'total_freight_all'
]

# Tỉ lệ giao hàng trễ (late delivery ratio)
temp = df_merged.groupby('customer_unique_id').apply(
    lambda x: (x['delivery_delay'] > 0).mean()
).reset_index()
temp.columns = ['customer_unique_id', 'late_delivery_ratio']
delivery_features = delivery_features.merge(temp, on='customer_unique_id', how='left')

print(f"   Đã tạo {len(delivery_features.columns)-1} logistics features")

   Đã tạo 11 logistics features


### 3. NHÓM 2: REVIEW

In [18]:
print("\n3. Nhóm 2: Review features")
review = df_merged.groupby('customer_unique_id').agg({
    'review_score': ['mean', 'std', 'min'],
    'num_comment_messages': lambda x: (~x.isna()).sum(),
    'num_comment_titles': lambda x: (~x.isna()).sum(),
    'days_to_answer': 'mean'
}).reset_index()
review.columns = ['customer_unique_id',
                  'avg_review_score', 'std_review_score', 'min_review_score',
                  'num_comments', 'num_titles', 'avg_days_to_answer']

low_review = df_merged.groupby('customer_unique_id')['review_score'].apply(lambda x: (x <= 2).mean())
review = review.merge(low_review.rename('low_review_ratio'), on='customer_unique_id')
bad_review_count = df_merged.groupby('customer_unique_id')['review_score'].apply(lambda x: (x <= 2).sum())
review = review.merge(bad_review_count.rename('num_bad_reviews'), on='customer_unique_id')
print(f"   -> {review.shape[1]-1} features")


3. Nhóm 2: Review features
   -> 8 features


### 4. NHÓM 3: HÀNH VI ( nhưng dùng total_price,...)

In [19]:
print("\n4. Nhóm 3: Behavior features")
df_merged['purchase_hour'] = df_merged['order_purchase_timestamp'].dt.hour
df_merged['purchase_weekday'] = df_merged['order_purchase_timestamp'].dt.dayofweek
df_merged['is_weekend'] = df_merged['purchase_weekday'].isin([5,6]).astype(int)
df_merged['is_night'] = df_merged['purchase_hour'].isin(range(20,24)).astype(int)

behavior = df_merged.groupby('customer_unique_id').agg({
    'num_products': ['mean', 'std'],
    'is_weekend': 'mean',
    'is_night': 'mean',
    'order_id': 'count'
}).reset_index()
behavior.columns = ['customer_unique_id',
                    'avg_items_per_order', 'std_items_per_order',
                    'weekend_purchase_ratio', 'night_purchase_ratio',
                    'total_orders']

# Khoảng cách giữa các lần mua
order_times = df_merged.sort_values(['customer_unique_id', 'order_purchase_timestamp'])
def get_gaps(group):
    if len(group) < 2:
        return pd.Series({'avg_gap': 0, 'std_gap': 0, 'max_gap': 0, 'trend_gap': 0})
    gaps = group.diff().dt.days.dropna()
    trend = np.polyfit(range(len(gaps)), gaps, 1)[0] if len(gaps) > 1 else 0
    return pd.Series({'avg_gap': gaps.mean(), 'std_gap': gaps.std(), 'max_gap': gaps.max(), 'trend_gap': trend})
gap_features = order_times.groupby('customer_unique_id')['order_purchase_timestamp'].apply(get_gaps).reset_index()
behavior = behavior.merge(gap_features, on='customer_unique_id')
print(f"   -> {behavior.shape[1]-1} features")


4. Nhóm 3: Behavior features
   -> 7 features


### 5. NHÓM 4: RFM (dùng total_payment thay vì payment_value)

In [20]:
print("\n5. Nhóm 4: RFM features")
rfm = df_merged.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (pd.Timestamp.now() - x.max()).days,
    'order_id': 'count',
    'total_payment': 'sum'
}).reset_index()
rfm.columns = ['customer_unique_id', 'recency', 'frequency', 'monetary']

rfm['recency_score'] = pd.qcut(rfm['recency'].rank(method='first'), q=4, labels=[4,3,2,1]).astype(int)
rfm['frequency_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['monetary_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['rfm_total'] = rfm['recency_score'] + rfm['frequency_score'] + rfm['monetary_score']

def rfm_segment(row):
    if row['recency_score']>=3 and row['frequency_score']>=3 and row['monetary_score']>=3:
        return 'champion'
    if row['recency_score']>=3 and row['frequency_score']>=2:
        return 'loyal'
    if row['recency_score']<=2 and row['frequency_score']<=2 and row['monetary_score']<=2:
        return 'at_risk'
    if row['recency_score']<=1:
        return 'churned'
    return 'promising'
rfm['rfm_segment'] = rfm.apply(rfm_segment, axis=1)
print(f"   -> {rfm.shape[1]-1} features")


5. Nhóm 4: RFM features
   -> 8 features
